# Loading of sessions

## Imports and session loading

In [ ]:
PROJECT_DIR = "./salient_kpms_1_ld7_jn24"  # Using a new directory for the full analysis
#MODEL_NAME = 'final_model_1.21e+04_idx4'
MODEL_NAME = 'final_model_1.20e+06_idx0'

In [ ]:
# --- IMPORTS & SETUP ---

# Enable autoreload (optional but good practice)
# %load_ext autoreload
# %autoreload 2

# Standard library & path setup
import sys
from pathlib import Path

# Data handling and analysis
import pandas as pd
import numpy as np
import pickle
import re

# Visualization
import matplotlib.pyplot as plt
import pprint

# Modeling pipeline
import keypoint_moseq as kpms
import jax

# --- 1. SETUP PROJECT PATH ---
print("Setting up project path...")
project_root = Path.cwd()
if not (project_root / "fiberphotometry").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
print(f"Project root added to path: {project_root}")

# --- 2. IMPORT CUSTOM MODULES ---
print("\nImporting custom project modules...")
try:
    # core Session class and loader
    from fiberphotometry.data.data_loading import Session, load_all_sessions

    # container for holding session data
    from fiberphotometry.data.session_loading import DataContainer

    # OFT-specific session population
    from fiberphotometry.data.oft_session_loading import populate_oft_containers

    # synchronizing different data streams in a session
    from fiberphotometry.data.session_synchronization import synchronize_session_data

    # KPMS pipeline stages
    from fiberphotometry.kpms.syllable_generator import generate_syllable_models
    from fiberphotometry.kpms.syllable_appender import extract_and_append_results
    from fiberphotometry.kpms.trimming import create_trimmed_coords_and_confs
    from fiberphotometry.kpms.batching import create_batches

    print("✅ All custom modules imported successfully.")
except ImportError as e:
    print(f"❌ Failed to import a custom module: {e}")
    print("   Please ensure your notebook is in the correct project directory.")


In [ ]:
PKL_NAME = 'big_run_frac_1_ld10_jn24'

In [ ]:
import pickle
import pandas as pd
import numpy as np

#load_filename = 'kpms_few_bp_150ms_frac_1_ld3_jn24.pkl'
load_filename = f'/home/fsp585/Desktop/FiberphotometryCode/tests/pkls/{PKL_NAME}.pkl'
print(f"--- 🚀 Loading complete analysis state from '{load_filename}'... ---")

with open(load_filename, 'rb') as f:
    loaded_state = pickle.load(f)

# unpack
sessions = loaded_state['sessions']
PROJECT_DIR_FULL = loaded_state.get('project_dir_full')
KAPPA = loaded_state.get('kappa')
video_frame_indexes = loaded_state['video_frame_indexes']
original_lengths = loaded_state['original_lengths']
final_model_names = loaded_state.get('final_model_names', loaded_state.get('model_names', []))

print(f"--- ✅ Base state loaded. Sessions: {len(sessions)}. Candidate models: {len(final_model_names)} ---")

# ---- materialize a single-model view into session.all_results ----
def _materialize_single_model_view(sessions, model_name):
    updated, missing = 0, []
    for s in sessions:
        store = getattr(s, 'kpms_results_by_model', None)
        if not store or model_name not in store:
            missing.append(getattr(s, 'trial_id', '?'))
            # ensure no stale legacy attribute lingers
            if hasattr(s, 'all_results'):
                delattr(s, 'all_results')
            continue

        res = store[model_name]

        # tolerate 'syllables' vs 'syllable' naming by creating a local alias if needed
        if 'syllable' not in res and 'syllables' in res:
            # make a shallow copy so we don't mutate the stored dict
            res = {**res, 'syllable': res['syllables']}

        # ensure no stale attr, then set the single-model projection
        if hasattr(s, 'all_results'):
            delattr(s, 'all_results')
        setattr(s, 'all_results', res)
        updated += 1
    return updated, missing

if 'MODEL_NAME' not in globals():
    raise NameError("Please set MODEL_NAME before loading (e.g., MODEL_NAME = final_model_names[0]).")

updated, missing = _materialize_single_model_view(sessions, MODEL_NAME)

print(f"✅ Materialized single-model view: {updated} sessions now have 'all_results' for '{MODEL_NAME}'.")
if missing:
    print(f"⚠️ {len(missing)} sessions lacked results for '{MODEL_NAME}': {missing[:6]}{' ...' if len(missing)>6 else ''}")

# quick sanity: do we actually have syllable vectors accessible like before?
any_ok = any(
    hasattr(s, 'all_results') and isinstance(s.all_results, dict) and
    (('syllable' in s.all_results) or ('syllables' in s.all_results))
    for s in sessions
)
print("🔎 Post-load check:", "OK ✅" if any_ok else "Issue ❌ — no sessions expose syllables via all_results")

In [ ]:
final_model_names

## Syllable syncing

In [ ]:
import pandas as pd

print("--- 🧠 Merging syllable data into synchronized DataFrames ---")

sessions_merged = 0
for i, s in enumerate(sessions):
    try:
        print(f"Processing session {i+1}/{len(sessions)}: {s.trial_id}")
        
        # --- 1. Check if the required data exists ---
        if not hasattr(s, 'all_results'):
            raise KeyError("Attribute 'all_results' not found on this session object.")
        if 'synchronized_data_tracking_0' not in s.dfs.data:
            raise KeyError("DataFrame 'synchronized_data_tracking_0' not found in s.dfs.data.")

        # --- 2. Get the target DataFrame and source data ---
        target_df = s.dfs.data['synchronized_data_tracking_0']
        syllable_array = s.all_results['syllable']

        # --- 3. Perform the length check as you requested ---
        if len(target_df) != len(syllable_array):
            # If lengths don't match, raise an error to be caught by the 'except' block
            raise ValueError(f"Length mismatch! DataFrame has {len(target_df)} rows, but syllable array has {len(syllable_array)} elements.")

        # --- 4. If checks pass, add the new column ---
        # This modifies the DataFrame directly within your session object
        target_df['syllable'] = syllable_array
        print(f"✅ Successfully added 'syllable' column.")
        sessions_merged += 1

    except (KeyError, ValueError) as e:
        # This will catch any errors raised above and print a warning,
        # allowing the loop to continue with the next session.
        print(f"❌ SKIPPING session {s.trial_id}: {e}")
    
    print("-" * 50)

print(f"\n--- ✅ Merge complete. {sessions_merged} out of {len(sessions)} sessions were updated. ---")


# --- Final Verification ---
# Let's check the first session to see if the new column is there.
print("\n--- 🔍 Final Verification of the first session ---")
if 'sessions' in locals() and 'syllable' in sessions[0].dfs.data['synchronized_data_tracking_0'].columns:
    print("Verification successful: 'syllable' column is present.")
    # Display the first few rows of the updated DataFrame
    #display(sessions[0].dfs.data['synchronized_data_tracking_0'].head())
else:
    print("Verification failed: 'syllable' column not found in the first session's DataFrame.")

In [ ]:
def build_used_frames_from_indexes(video_frame_indexes, original_lengths, out_dir):
    """
    Build *_used_frames.pkl files from:
      - video_frame_indexes: dict[str -> np.ndarray of absolute frame indices]
      - original_lengths: dict[str -> int] (total length per base video)
    Writes one file per base: {base}_used_frames.pkl in out_dir.
    Returns a dict: {filename: mask_np}.
    """
    import re, pickle
    from collections import defaultdict
    from pathlib import Path
    import numpy as np

    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    # Group frame-index arrays by base (strip optional _partN at the end)
    def strip_part(name: str) -> str:
        return re.sub(r"_part\d+$", "", name)

    grouped = defaultdict(list)
    for k, idx in video_frame_indexes.items():
        base = strip_part(k)
        grouped[base].append(np.asarray(idx, dtype=np.int64))

    used_dict = {}
    for base, length in original_lengths.items():
        L = int(length)
        mask = np.zeros(L, dtype=np.uint8)

        part_arrays = grouped.get(base, [])
        if not part_arrays:
            # No parts found for this base — keep all zeros but warn
            print(f"[warn] no indexes for base='{base}'. Writing all-zeros mask of len={L}.")

        for arr in part_arrays:
            # safety: clamp to [0, L-1]
            valid = (arr >= 0) & (arr < L)
            if not np.all(valid):
                bad = int((~valid).sum())
                print(f"[warn] {base}: {bad} indices out of bounds (kept {valid.sum()}/{arr.size}).")
                arr = arr[valid]
            mask[arr] = 1  # OR across parts

        fname = f"{base}_used_frames.pkl"
        with open(out_dir / fname, "wb") as f:
            pickle.dump(mask, f)
        used_dict[fname] = mask

        # tiny summary
        print(f"[ok] wrote {fname}  len={L}, used={int(mask.sum())}, ratio={mask.mean():.4f}")

    return used_dict


In [ ]:
out_path = "/home/fsp585/Desktop/FiberphotometryCode/tests/used_frames"
new_used = build_used_frames_from_indexes(video_frame_indexes, original_lengths, out_path)

In [ ]:
sessions[0].kpms_h5_path

In [ ]:
len(sessions[0].all_results['syllable'])

In [ ]:
# Parse your pasted log, compare to sessions[*].all_results['syllable'] length

import os, re

used_log = """
[ok] wrote Arena1234_CenterTopCamVid_2024-12-13T12_20_06_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=319144, used=220475, ratio=0.6908
[ok] wrote Arena1234_CenterTopCamVid_2024-12-13T12_20_06_bottom_rightDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=319144, used=218463, ratio=0.6845
[ok] wrote Arena1234_CenterTopCamVid_2024-12-13T12_20_06_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=319144, used=231374, ratio=0.7250
[ok] wrote Arena1234_CenterTopCamVid_2024-12-13T13_37_45_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=294672, used=196653, ratio=0.6674
[ok] wrote Arena1234_CenterTopCamVid_2024-12-13T13_37_45_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=294672, used=222797, ratio=0.7561
[ok] wrote Arena1234_CenterTopCamVid_2024-12-13T14_50_39_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=305984, used=221848, ratio=0.7250
[ok] wrote Arena1234_CenterTopCamVid_2024-12-13T14_50_39_bottom_rightDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=305984, used=222645, ratio=0.7276
[ok] wrote Arena1234_CenterTopCamVid_2024-12-13T14_50_39_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=305984, used=221549, ratio=0.7241
[ok] wrote Arena1234_CenterTopCamVid_2024-12-16T11_27_19_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=316028, used=206811, ratio=0.6544
[ok] wrote Arena1234_CenterTopCamVid_2024-12-16T11_27_19_bottom_rightDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=316028, used=218219, ratio=0.6905
[ok] wrote Arena1234_CenterTopCamVid_2024-12-16T11_27_19_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=316028, used=253732, ratio=0.8029
[ok] wrote Arena1234_CenterTopCamVid_2024-12-16T12_42_27_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=310401, used=219502, ratio=0.7072
[ok] wrote Arena1234_CenterTopCamVid_2024-12-16T12_42_27_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=310401, used=226268, ratio=0.7290
[ok] wrote Arena1234_CenterTopCamVid_2024-12-16T14_05_53_bottom_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=302304, used=221038, ratio=0.7312
[ok] wrote Arena1234_CenterTopCamVid_2024-12-16T14_05_53_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=302304, used=223005, ratio=0.7377
[ok] wrote Arena1234_CenterTopCamVid_2024-12-16T15_14_26_top_leftDLC_HrnetW32_dataset_fixDec16shuffle10_detector_020_snapshot_020_used_frames.pkl  len=276529, used=220510, ratio=0.7974
""".strip()

# Build mapping: base filename (without _used_frames.pkl) -> {'len':..., 'used':..., 'ratio':...}
used_map = {}
pat = re.compile(
    r"wrote\s+(?P<base>.+?)_used_frames\.pkl\s+len=(?P<len>\d+),\s*used=(?P<used>\d+),\s*ratio=(?P<ratio>[\d.]+)"
)
for line in used_log.splitlines():
    m = pat.search(line)
    if m:
        d = m.groupdict()
        used_map[d["base"]] = {
            "len": int(d["len"]),
            "used": int(d["used"]),
            "ratio": float(d["ratio"]),
        }

# Helper to match a session to the correct log entry
def match_base_for_session(h5_path: str):
    base = os.path.basename(h5_path).replace(".h5", "")
    if base in used_map:
        return base
    # fallback: fuzzy match by containment if names differ slightly
    candidates = [b for b in used_map if b in base or base in b]
    return candidates[0] if len(candidates) == 1 else None

rows = []
missing = []
extras = set(used_map.keys())

for i, s in enumerate(sessions):
    key = match_base_for_session(s.kpms_h5_path)
    syl_len = len(s.all_results["syllable"])
    if key is None:
        missing.append((i, os.path.basename(s.kpms_h5_path), syl_len))
        continue
    info = used_map[key]
    expected = info["len"]
    diff = syl_len - expected
    rel = (abs(diff) / expected) if expected else float("inf")
    rows.append({
        "idx": i,
        "base": key,
        "expected_len": expected,
        "syllable_len": syl_len,
        "diff": diff,
        "rel_diff": rel,
        "used_count": info["used"],
        "used_ratio": info["ratio"],
    })
    if key in extras:
        extras.remove(key)

# Sort by absolute difference, largest first
rows.sort(key=lambda r: abs(r["diff"]), reverse=True)

# Pretty print
def fmt_row(r):
    return (f"[{r['idx']:>2}] {r['base']}  "
            f"expected={r['expected_len']:,}  syllable={r['syllable_len']:,}  "
            f"diff={r['diff']:+}  rel={r['rel_diff']:.6f}  used={r['used_count']:,}  ratio={r['used_ratio']:.4f}")

print("=== used_frames vs all_results['syllable'] length check ===")
if rows:
    for r in rows:
        flag = "  <-- MISMATCH" if r["diff"] != 0 else ""
        print(fmt_row(r) + flag)
else:
    print("No matches found between sessions and the pasted log.")

# Summary
mismatches = [r for r in rows if r["diff"] != 0]
print("\nSummary:")
print(f"  matched: {len(rows)}")
print(f"  exact matches: {len(rows) - len(mismatches)}")
print(f"  mismatches: {len(mismatches)}")

if mismatches:
    max_abs = max(abs(r["diff"]) for r in mismatches)
    avg_abs = sum(abs(r["diff"]) for r in mismatches) / len(mismatches)
    print(f"  max |diff|: {max_abs}")
    print(f"  avg |diff|: {avg_abs:.2f}")

if missing:
    print("\nSessions with no corresponding log entry:")
    for idx, name, syl_len in missing:
        print(f"  [{idx:>2}] {name}  syllable_len={syl_len:,}")

if extras:
    print("\nLog entries not found among sessions (by base name):")
    for b in sorted(extras):
        print(f"  {b}")


In [ ]:
sessions[0].all_results

In [ ]:
# Build dict {simplified_fname -> DataFrame} from sessions[*].all_results and save as dfs_by_fname.pkl

import os, re, pickle
import pandas as pd
import numpy as np

def simplify_name(h5_path: str) -> str:
    base = os.path.basename(h5_path)
    if base.endswith(".h5"):
        base = base[:-3]
    # Remove everything from 'DLC' (case-insensitive) to the end, and any trailing underscore
    base = re.sub(r'(?i)DLC.*$', '', base).rstrip('_')
    return base

def results_to_df(ar: dict) -> pd.DataFrame:
    # Target length = length of 'syllable' if present, else first array-like value
    if 'syllable' in ar and hasattr(ar['syllable'], '__len__'):
        n = len(ar['syllable'])
    else:
        n = next((len(v) for v in ar.values() if hasattr(v, '__len__')), None)
        if n is None:
            raise ValueError("all_results has no array-like values")

    cols = {}
    for k, v in ar.items():
        a = np.asarray(v)
        if a.ndim == 1:
            colnames = [k]
            arrays = [a]
        elif a.ndim == 2:
            # Name 2D columns as k_0, k_1, ...
            colnames = [f"{k}_{j}" for j in range(a.shape[1])]
            arrays = [a[:, j] for j in range(a.shape[1])]
        else:
            # Skip higher-dim entries
            continue

        for name, arr in zip(colnames, arrays):
            arr = np.asarray(arr)
            # Pad/truncate to length n
            if len(arr) < n:
                pad = np.full(n - len(arr), np.nan, dtype=float if not np.issubdtype(arr.dtype, np.floating) else arr.dtype)
                arr = np.concatenate([arr, pad])
            elif len(arr) > n:
                arr = arr[:n]
            cols[name] = arr

    # Friendlier names for centroid if present
    if 'centroid_0' in cols and 'centroid_1' in cols:
        cols['centroid_x'] = cols.pop('centroid_0')
        cols['centroid_y'] = cols.pop('centroid_1')

    return pd.DataFrame(cols)

dfs_by_fname = {}
dups = []
for s in sessions:
    key = simplify_name(s.kpms_h5_path)
    if key in dfs_by_fname:
        dups.append(key)
        i = 2
        alt = f"{key}__{i}"
        while alt in dfs_by_fname:
            i += 1
            alt = f"{key}__{i}"
        key = alt
    dfs_by_fname[key] = results_to_df(s.all_results)

out_path = f"/home/fsp585/Desktop/FiberphotometryCode/tests/used_frames/dfs_by_fname_{PKL_NAME}.pkl"
with open(out_path, "wb") as f:
    pickle.dump(dfs_by_fname, f)

print(f"Saved {len(dfs_by_fname)} DataFrames to {out_path}")
if dups:
    print("Duplicate base names encountered; suffixed entries:", dups[:5], "..." if len(dups) > 5 else "")
# Example to load later:
# with open(out_path, "rb") as f:
#     dfs = pickle.load(f)
#     display(dfs['Arena1234_CenterTopCamVid_2024-12-16T12_42_27_bottom_left'].head())


In [ ]:
for k, v in dfs_by_fname.items():
    print(k, len(v), v.columns)

## Model loading and more sanity checks

In [ ]:
# # define your ten model names
# model_names = [f'final_model_1.20e+06_idx{i}' for i in range(5)]

# # compute EML scores and their standard errors
# eml_scores, eml_std_errs = kpms.expected_marginal_likelihoods(PROJECT_DIR, model_names)

# # pick the best model
# best_model_namxe = model_names[np.argmax(eml_scores)]
# print(f"Best model: {best_model_name}")

# # visualize the EML scores across the ensemble
# kpms.plot_eml_scores(eml_scores, eml_std_errs, model_names)


In [ ]:
import keypoint_moseq as kpms
model, data, metadata, current_iter = kpms.load_checkpoint(PROJECT_DIR, MODEL_NAME)
results = kpms.extract_results(model, metadata, PROJECT_DIR, MODEL_NAME, save_results=False)

In [ ]:
# model_names = [f'final_model_1.20e+06_idx{i}' for i in range(5)]

# for model_name in model_names:
#     # load the specific checkpoint
#     model, data, metadata, current_iter = kpms.load_checkpoint(PROJECT_DIR, model_name)
    
#     # extract results for this model
#     results = kpms.extract_results(model, metadata, PROJECT_DIR, model_name, save_results=True)


In [ ]:
config = kpms.load_config(PROJECT_DIR)

In [ ]:
import os
import h5py
import numpy as np
import jax.numpy as jnp
from jax_moseq.utils import unbatch
from jax_moseq.models.keypoint_slds import estimate_coordinates

# compute the estimated coordinates
Y_est = estimate_coordinates(
    jnp.array(model['states']['x']),
    jnp.array(model['states']['v']),
    jnp.array(model['states']['h']),
    jnp.array(model['params']['Cd'])
)

# generate a dictionary with reconstructed coordinates for each recording
coordinates_est = unbatch(Y_est, *metadata)

In [ ]:
config['bodyparts'] = config['use_bodyparts']

In [ ]:
centroids, headings = kpms.get_centroids_headings(coordinates_est, **config)

## Group assignment and syllable labeling

In [ ]:
import re

# metadata_names is the list/array of all result names (e.g., metadata[0])
metadata_names = metadata[0]

for sess in sessions:
    sess.result_names = []
    h5_path = sess.kpms_h5_path
    for name in metadata_names:
        # strip "_part#" only for matching
        base = re.sub(r'_part\d+$', '', name)
        if base in h5_path and name not in sess.result_names:
            # append the full original name (with part suffix)
            sess.result_names.append(name)

In [ ]:
import pandas as pd

# build a list of dicts
rows = []
for sess in sessions:
    for name in sess.result_names:
        rows.append({
            "name": name,
            "group": sess.genotype
        })

# make a DataFrame
index_df = pd.DataFrame(rows)

# (optionally) save it back out so your interactive widget picks it up
index_df.to_csv(os.path.join(PROJECT_DIR, "index.csv"), index=False)

# inspect
print(index_df.head())


In [ ]:
moseq_df = kpms.compute_moseq_df(PROJECT_DIR, MODEL_NAME, smooth_heading=True)

In [ ]:
moseq_df

In [ ]:
stats_df = kpms.compute_stats_df(
    PROJECT_DIR,
    MODEL_NAME,
    moseq_df,
    min_frequency=0.005,  # threshold frequency for including a syllable in the dataframe
    groupby=["group", "name"],  # column(s) to group the dataframe by
    fps=60,
)  # frame rate of the video from which keypoints were inferred

stats_df

In [ ]:
sessions[0].kpms_h5_path

## Plotting setup and more checks

In [ ]:
from fiberphotometry.config import PLOTTING_CONFIG
from fiberphotometry.processing.plotting_setup import PlottingSetup

In [ ]:
PlottingSetup(**PLOTTING_CONFIG['cpt']).apply_plotting_setup_to_sessions(sessions)

## Built-in analyzis

In [ ]:
kpms.plot_syll_stats_with_sem(
    stats_df,
    PROJECT_DIR,
    MODEL_NAME,
    plot_sig=True,  # whether to mark statistical significance with a star
    thresh=0.05,  # significance threshold
    stat="frequency",  # statistic to be plotted (e.g. 'duration' or 'velocity_px_s_mean')
    order="stat",  # order syllables by overall frequency ("stat") or degree of difference ("diff")
    ctrl_group="a",  # name of the control group for statistical testing
    exp_group="b",  # name of the experimental group for statistical testing
    figsize=(8, 4),  # figure size
    groups=stats_df["group"].unique(),  # groups to be plotted
);


In [ ]:
normalize = "bigram"  # normalization method ("bigram", "rows" or "columns")

trans_mats, usages, groups, syll_include = kpms.generate_transition_matrices(
    PROJECT_DIR,
    MODEL_NAME,
    normalize=normalize,
    min_frequency=0.010,  # minimum syllable frequency to include
)

kpms.visualize_transition_bigram(
    PROJECT_DIR,
    MODEL_NAME,
    groups,
    trans_mats,
    syll_include,
    normalize=normalize,
    show_syllable_names=True,  # label syllables by index (False) or index and name (True)
)


In [ ]:

kpms.plot_transition_graph_group(
    PROJECT_DIR,
    MODEL_NAME,
    groups,
    trans_mats,
    usages,
    syll_include,
    layout="circular",  # transition graph layout ("circular" or "spring")
    show_syllable_names=False,  # label syllables by index (False) or index and name (True)
)


In [ ]:
kpms.plot_transition_graph_group(
    PROJECT_DIR,
    MODEL_NAME,
    groups,
    trans_mats,
    usages,
    syll_include,
    layout="circular",  # transition graph layout ("circular" or "spring")
    show_syllable_names=True,  # label syllables by index (False) or index and name (True)
)


In [ ]:
kpms.plot_transition_graph_difference(
    PROJECT_DIR, MODEL_NAME, groups, trans_mats, usages, syll_include, layout="circular", show_syllable_names=True
)  # transition graph layout ("circular" or "spring")

## PETHs

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import sem
from scipy.signal import savgol_filter
import os

# --- define output directory based on model name ---
fig_base_dir = os.path.join("syllable_figs", MODEL_NAME)
os.makedirs(fig_base_dir, exist_ok=True)

In [ ]:
# --- Params (same as before) ---
SAMPLING_RATE = 80
WINDOW_SECONDS = 10
SMOOTHING_WINDOW = 25
SMOOTHING_POLY_ORDER = 3
CONFIDENCE_LEVEL = 1.96
WINDOW_SAMPLES = int(WINDOW_SECONDS * SAMPLING_RATE)
TOTAL_SAMPLES = 2 * WINDOW_SAMPLES
SIGNAL_COLS = ['signal_0_470_dff_z', 'signal_1_470_dff_z']

# Optional: only plot syllables that have at least this many total traces across sessions
MIN_TRACES_PER_SYLLABLE = 10  # set to 0 to include all

# --- Helpers ---
def _interp_then_smooth(x, win, poly):
    """Linear-interpolate NaNs then SavGol smooth. Handles short traces gracefully."""
    x = np.asarray(x, dtype=float)
    n = x.size

    # Interpolate NaNs
    if np.isnan(x).any():
        idx = np.arange(n)
        good = ~np.isnan(x)
        if good.any():
            # Edge-hold at ends
            left_val = x[good][0]
            right_val = x[good][-1]
            x[~good] = np.interp(idx[~good], idx[good], x[good])
            # ensure fully filled (interp covers interior; endpoints ok)
            if np.isnan(x[0]): x[0] = left_val
            if np.isnan(x[-1]): x[-1] = right_val
        else:
            # all NaNs → return zeros
            return np.zeros_like(x)

    # Ensure odd window and not longer than series
    w = min(win, n - (n % 2 == 0))  # if n is even, w<=n-1
    if w < 5:  # too small to smooth meaningfully
        return x
    if w % 2 == 0:
        w -= 1
    if w <= poly:
        poly = max(1, min(poly, w - 2))
    return savgol_filter(x, w, poly)

def process_session_data(session):
    """Extract, smooth, baseline-correct traces around syllable onsets for one session."""
    df = session.dfs.data['synchronized_data_tracking_0']
    syll = df['syllable']

    # Onsets = points where syllable changes (and current is not NaN)
    prev = syll.shift(1)
    onsets = syll[(syll.notna()) & (syll != prev)]

    # Prepare containers
    syllable_traces = {}  # {syllable_id: {col: [trace, ...], ...}, ...}

    for idx, syl in onsets.items():
        start = idx - WINDOW_SAMPLES
        end = idx + WINDOW_SAMPLES
        if start < 0 or end > len(df):
            continue

        # ensure dicts exist
        if syl not in syllable_traces:
            syllable_traces[syl] = {col: [] for col in SIGNAL_COLS}

        for col in SIGNAL_COLS:
            trace = df.iloc[start:end][col].to_numpy()
            if trace.shape[0] != TOTAL_SAMPLES:
                continue
            smoothed = _interp_then_smooth(trace, SMOOTHING_WINDOW, SMOOTHING_POLY_ORDER)
            baseline = smoothed[WINDOW_SAMPLES]  # sample at onset
            syllable_traces[syl][col].append(smoothed - baseline)

    return syllable_traces

# --- Process all sessions ---
processed_list = [process_session_data(s) for s in sessions]

# Count total traces per syllable (across signals + sessions) for filtering/ordering
syll_counts = {}
for proc in processed_list:
    for syl, d in proc.items():
        syll_counts[syl] = syll_counts.get(syl, 0) + sum(len(v) for v in d.values())

# Decide which syllables to plot
all_syllables = sorted([s for s, c in syll_counts.items() if c >= MIN_TRACES_PER_SYLLABLE])
print(f"Syllables to plot (min traces {MIN_TRACES_PER_SYLLABLE}):", all_syllables)

# Regions / hemispheres from metadata
regions = sorted({region for s in sessions for region, _, _ in s.brain_regions})
hemispheres = sorted({
    s.signal_meta["_".join(col.split("_")[:2])][1]
    for s in sessions
    for col in SIGNAL_COLS
    if "_".join(col.split("_")[:2]) in s.signal_meta
})
time_vector = np.linspace(-WINDOW_SECONDS, WINDOW_SECONDS, TOTAL_SAMPLES)

# --- Plot ---
for syl_idx, syllable in enumerate(all_syllables):
    hemis_order = ['left', 'right']
    n_rows, n_cols = 2, len(regions)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols + 2, 8))
    fig.suptitle(f"Syllable Onset: ID {int(syllable)}", fontsize=18)

    for row_idx, hemisphere in enumerate(hemis_order):
        for col_idx, region in enumerate(regions):
            ax = axes[row_idx, col_idx] if n_rows > 1 else axes[col_idx]
            traces_by_genotype = {'KI': [], 'WT': []}

            for s, proc_data in zip(sessions, processed_list):
                meta = s.signal_meta
                for col in SIGNAL_COLS:
                    prefix = "_".join(col.split("_")[:2])
                    if prefix not in meta:
                        continue
                    region_meta, hemi_meta = meta[prefix][0], meta[prefix][1]
                    if region_meta == region and hemi_meta == hemisphere:
                        traces = proc_data.get(syllable, {}).get(col, [])
                        traces_by_genotype[s.genotype].extend(traces)

            for geno, traces in traces_by_genotype.items():
                if not traces:
                    continue
                valid = [t for t in traces if not np.isnan(t).any()]
                if not valid:
                    continue
                arr = np.stack(valid)
                m = arr.mean(axis=0)
                e = sem(arr, axis=0)
                lower, upper = m - CONFIDENCE_LEVEL * e, m + CONFIDENCE_LEVEL * e

                color = 'darkblue' if geno == 'KI' else 'green'
                ax.plot(time_vector, m, label=f"{geno} (n={len(valid)})", color=color)
                ax.fill_between(time_vector, lower, upper, color=color, alpha=0.2)

            ax.set_title(f"{region} | {hemisphere}", fontsize=13)
            ax.axvline(0, color='crimson', linestyle='--', lw=2)
            ax.axhline(0, color='grey', linestyle=':', lw=1)
            if row_idx == n_rows - 1:
                ax.set_xlabel("Time from Onset (s)")
            if col_idx == 0:
                ax.set_ylabel("Normalized Signal (z-score)")
            ax.grid(True, linestyle=':')
            ax.legend(fontsize='small')

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    
    out_path = os.path.join(fig_base_dir, f"{int(syllable)}.png")
    plt.savefig(out_path)
    plt.close(fig)   # good practice when saving lots of figs
    print(f"💾 Saved: {out_path}")

## Creation of "mother" dataframe

In [ ]:
list(moseq_df.name.unique())

In [ ]:
sessions[1].kpms_h5_path

In [ ]:
sessions[1].dfs.data['synchronized_data_tracking_0']

In [ ]:
moseq_df

In [ ]:
# --- helpers -----------------------------------------
def _base_from_key(k: str) -> str:
    return k.split('_part')[0] if '_part' in k else k

def _part_num_or_none(k: str):
    if '_part' not in k:
        return None
    return int(k.split('_part')[-1])

def _base_from_h5_path(h5_path: str) -> str:
    fname = h5_path.rsplit('/', 1)[-1]
    return fname.rsplit('.', 1)[0]

# --- 1) Build per-session USED row arrays (concat parts in order) ------------
base_to_used_rows = {}
for k, arr in video_frame_indexes.items():
    b = _base_from_key(k)
    base_to_used_rows.setdefault(b, []).append((k, _part_num_or_none(k), arr))

for b, items in base_to_used_rows.items():
    if len(items) == 1 and items[0][1] is None:
        base_to_used_rows[b] = items[0][2]
    else:
        items_sorted = sorted(items, key=lambda t: t[1])
        base_to_used_rows[b] = np.concatenate([t[2] for t in items_sorted])

video_bases = set(base_to_used_rows.keys())

# --- 2) Tag MoSeq with session_base (from the filename embedded in `name`) ---
import re
pattern = "(" + "|".join(re.escape(b) for b in sorted(video_bases)) + ")"
moseq_tagged = moseq_df.copy()
moseq_tagged["session_base"] = moseq_tagged["name"].str.extract(pattern, expand=False)

# Optional: quick flag if a MoSeq row’s frame_index would be out of bounds for its session
def _flag_in_video_index(group):
    used = base_to_used_rows.get(group.name, None)
    if used is None or len(used) == 0:
        group["in_video_index"] = False
    else:
        n = len(used)
        fi = group["frame_index"].to_numpy()
        group["in_video_index"] = (fi >= 0) & (fi < n)
    return group

moseq_tagged = (
    moseq_tagged
    .dropna(subset=["session_base"])
    .groupby("session_base", group_keys=False)
    .apply(_flag_in_video_index)
)

# --- 3) Build a single, annotated signal DF with a MoSeq-like index ----------
# For each session:
#   - get its sync DF
#   - compute inverse map: current_h5_row -> local MoSeq frame_index (position in used array)
#   - add `session_base` and `moseq_frame_index` columns
sync_chunks = []
for sess in sessions:
    base = _base_from_h5_path(sess.kpms_h5_path)
    sync_df = sess.dfs.data.get("synchronized_data_tracking_0", None)
    if (base not in base_to_used_rows) or (sync_df is None):
        continue

    used = base_to_used_rows[base]  # array of current_h5_row values used by MoSeq
    # inverse map: for each used current_h5_row, what is its local MoSeq index?
    inv = pd.Series(np.arange(len(used), dtype=np.int64), index=used)

    s = sync_df.copy()
    if "current_h5_row" not in s.columns:
        s = s.reset_index().rename(columns={"index": "current_h5_row"})

    s["session_base"] = base
    s["moseq_frame_index"] = s["current_h5_row"].map(inv).astype("Int64")  # NaN where signal row wasn’t used by MoSeq
    sync_chunks.append(s)

sync_all = pd.concat(sync_chunks, ignore_index=True) if sync_chunks else pd.DataFrame()

# --- 4) Merge: MoSeq (left) x annotated signal on (session_base, frame_index) --
# Guarantees len(output) == len(moseq_df); unmatched signal rows are not included.
merged_all = moseq_tagged.merge(
    sync_all,
    left_on=["session_base", "frame_index"],
    right_on=["session_base", "moseq_frame_index"],
    how="left",
    suffixes=("", "_fp"),
    indicator=True
)
merged_all["had_signal_match"] = merged_all["_merge"].eq("both")
merged_all.drop(columns=["_merge"], inplace=True)

# --- 5) (optional) tidy columns ------------------------------------------------
preferred_left = [
    "session_base", "name", "group", "syllable", "onset",
    "frame_index", "in_video_index", "had_signal_match"
]
left_cols = [c for c in preferred_left if c in merged_all.columns]
other_cols = [c for c in merged_all.columns if c not in left_cols]
merged_all = merged_all[left_cols + other_cols]

# Optional stable sort (per session, by MoSeq order)
# merged_all.sort_values(["session_base", "frame_index"], inplace=True, kind="stable")


In [ ]:
moseq_df

In [ ]:
merged_all.columns

## Syllable entropy

Does the KI have less behavioral diversity and longer bouts?

## Transition entropy

Does the KI show less sequence entropy?

## Event-aligned DA at onset

How does the DS/VS act during exploratory vs repetitive syllables across genotypes?

## Kinematics of syllables

How does the execution of syllables differ across genotypes? Is there a coupling between the DA of a syllable and the ensuing vigor of said syllable instance?

## Can kinematics and syllable signature predict said syllable?

## Heatmap of syllable execution

What syllables are (not) center biased and which are more neutral? Do syllable signatures change depending on location and genotype?

## Linear mixed models

Using mixed models, can we get a more clear indication of significant differences in genotype response?

##